# 02 - Multi-Agent Teams with AutoGen

## Scenario: Northstar Incident Response Team

A single agent can become confused when given too many tools and complex instructions. By splitting responsibilities into a multi-agent team, we can have an `Investigator` focused entirely on evidence gathering, and a `Reviewer` focused entirely on safety policies.

In this module, we will use **Microsoft AutoGen** to build a Group Chat where multiple agents collaborate to solve a checkout incident.

In [ ]:
import os
from autogen import ConversableAgent, GroupChat, GroupChatManager

# Note: Set your OPENAI_API_KEY environment variable. 
# We use a mock config here for learning purposes if you don't have one.
llm_config = {"model": "gpt-4o", "api_key": os.environ.get("OPENAI_API_KEY", "dummy-key")}

# 1. Define the Agents
investigator = ConversableAgent(
    name="Investigator",
    system_message="You are the Northstar Incident Investigator. You find the root cause of issues. You do not propose solutions, you only report facts.",
    llm_config=llm_config,
)

reviewer = ConversableAgent(
    name="Reviewer",
    system_message="You are the Northstar Safety Reviewer. You read the investigator's facts and propose a safe mitigation. You NEVER authorize executing code in production directly.",
    llm_config=llm_config,
)

commander = ConversableAgent(
    name="Commander",
    system_message="You are the Incident Commander. You review the mitigation plan from the Reviewer. If it is safe, you output the word 'APPROVED'.",
    llm_config=llm_config,
    is_termination_msg=lambda msg: "APPROVED" in msg.get("content", ""),
)


## 1. Setting up the Group Chat

In AutoGen, agents talk to each other in a shared room managed by a `GroupChatManager`. You can control who speaks next, or let the LLM auto-select the next speaker.

In [ ]:
# We use round_robin to force a strict order: Investigator -> Reviewer -> Commander
groupchat = GroupChat(
    agents=[investigator, reviewer, commander],
    messages=[],
    max_round=6,
    speaker_selection_method="round_robin"
)

manager = GroupChatManager(groupchat=groupchat, llm_config=llm_config)


## 2. Running the Chat

When you initiate the chat, the agents will take turns until the termination condition is met (e.g. Commander says 'APPROVED').

In [ ]:
# Wrap in try/except for mock environments
try:
    chat_result = commander.initiate_chat(
        manager,
        message="Investigate the EU checkout failure incident, propose a mitigation, and approve it."
    )
    print("Chat finished.")
except Exception as e:
    print(f"API Error (Expected if using a dummy key): {e}")


## Watch For

- **Endless Debates**: Multi-agent systems can get stuck arguing with each other. Always use `max_round` and clear termination messages.
- **Role Bleed**: If system prompts aren't strict enough, the Investigator might try to approve the fix itself.
- **Cost Multipliers**: Every message in a group chat is sent to every agent, meaning token costs grow quadratically.

## Checkpoint

**1. What is the role of the `GroupChatManager` in AutoGen?**
- A) It provides the tools to the agents.
- B) It holds the conversation history and selects the next speaker based on the rules.
- C) It connects to the database.
- D) It generates the final report.

**2. Why should you use `speaker_selection_method="round_robin"` in a strict incident response pipeline?**
- A) Because the LLM is not smart enough to auto-select.
- B) To enforce a strict compliance order (investigate -> review -> approve) without unpredictable LLM routing.
- C) It saves memory.
- D) It prevents hallucinated tools.
